# Use Model Context Protocol (MCP) as tools with Strands Agent

## Overview
The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) is an open protocol that standardizes how applications provide context to Large Language Models (LLMs). Strands AI SDK integrates with MCP to extend agent capabilities through external tools and services.

MCP enables communication between agents and MCP servers that provide additional tools. The Strands Agent SDK includes built-in support for connecting to MCP servers and using their tools.

In this example we will show you how to use MCP tools on your Strands Agent. We will use the [AWS Documentation MCP server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server/) which provides tools to access AWS documentation, search for content, and get recommendations. This MCP server has 3 main features:

- **Read Documentation**: Fetch and convert AWS documentation pages to markdown format
- **Search Documentation**: Search AWS documentation using the official search API
- **Recommendations**: Get content recommendations for AWS documentation pages



## Agent Details
<div style="float: left; margin-right: 20px;">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|Feature used        |MCP Tools                                          |
|Agent Structure     |Single agent architecture                          |

</div>

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## Key Features
* **Single agent architecture**: this example creates a single agent that interacts with MCP tools
* **MCP tools**: Integration of MCP tools with your agent

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock

Let's now install the requirement packages for our Strands Agent agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [ ]:
import threading
import time
from datetime import timedelta

from mcp import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamablehttp_client
from mcp.server import FastMCP
from strands import Agent
from strands.tools.mcp import MCPClient


### Connect to MCP server using stdio transport

[Transports](https://modelcontextprotocol.io/specification/2025-03-26/basic/transports) in MCP provide the foundations for communication between clients and servers. It handles the underlying mechanics of how messages are sent and received. At the moment there are three standards transport implementations built-in in MCP:

- **Standard Input/Output (stdio)**: enables communication through standard input and output streams. It is particularly useful for local integrations and command-line tools
- **Streamable HTTP**: this replaces the HTTP+SSE transport from previous protocol version. In the Streamable HTTP transport, the server operates as an independent process that can handle multiple client connections. This transport uses HTTP POST and GET requests. Server can optionally make use of Server-Sent Events (SSE) to stream multiple server messages. This permits basic MCP servers, as well as more feature-rich servers supporting streaming and server-to-client notifications and requests.
- **SSE**: legacy transport for HTTP-based MCP servers that use Server-Sent Events transport  

Overall, you should use stdio for building command-line tools, implementing local integrations and working with shell scripts. You should use Streamable HTTP transports when you need a flexible and efficient way for AI agents to communicate with tools and services, especially when dealing with stateless communication or when minimizing resource usage is crucial.

You can also use **custom transports** implementation for your specific needs. 


Let's now connect to the MCP server using stdio transport. First of all, we will use the class `MCPClient` to connect to the [AWS Documentation MCP Server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server/). This server provides tools to access AWS documentation, search for content, and get recommendations.

In [3]:
# Connect to an MCP server using stdio transport
stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `stdio_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [12]:
# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    tools = stdio_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="us.amazon.nova-2-lite-v1:0",
        tools=tools)

    response = agent("What is Amazon Bedrock pricing model. Be concise.")


Tool #1: search_documentation

Tool #2: read_sections
Amazon Bedrock uses a token-based pricing model where costs are driven by model inference. Each API call processes tokens (units of text the model reads/generates), and your cost depends on:

- **Model** - Different models have different per-token rates  
- **Token type** - Input tokens, output tokens, cache read tokens, and cache write tokens each priced separately  
- **Routing** - Cross-region inference costs more than in-region requests  
- **Inference tier** - On-demand, provisioned throughput, and batch inference have different pricing structures  

For current pricing details, visit the official Amazon Bedrock pricing page.

### Connect to MCP server using Streamable HTTP

Let's now connect to the MCP server using Streamable HTTP transport. First let's start a simple MCP server using Streamable HTTP transport. 

For this example we will create our own MCP server. The architecture will look as following

<div style="text-align:center">
    <img src="images/architecture_2.png" width="65%" />
</div>

In [13]:
# Create an MCP server
mcp = FastMCP("Calculator Server")

# Define a tool


@mcp.tool(description="Calculator tool which performs calculations")
def calculator(x: int, y: int) -> int:
    return x + y


@mcp.tool(description="This is a long running tool")
def long_running_tool(name: str) -> str:
    time.sleep(25)
    return f"Hello {name}"


def main():
    mcp.run(transport="streamable-http", mount_path="mcp")

Let's now start a thread with the `streamable-http` server

In [14]:
thread = threading.Thread(target=main)
thread.start()

INFO:     Started server process [258451]
INFO:     Waiting for application startup.


[04/25/26 15:21:51] INFO     StreamableHTTP session manager started                  ]8;id=14503277;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=14503278;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


#### Integrating Streamable HTTP client with Agent

Now let's use `streamablehttp_client` integrate this server with a simple agent. 

In [15]:
def create_streamable_http_transport():
    return streamablehttp_client("http://localhost:8000/mcp")


streamable_http_mcp_client = MCPClient(create_streamable_http_transport)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `streamable_http_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [17]:
with streamable_http_mcp_client:
    tools = streamable_http_mcp_client.list_tools_sync()

    agent = Agent(
        model="us.amazon.nova-2-lite-v1:0",
        tools=tools)

    response = str(agent("What is 2 + 2?"))

[04/25/26 15:22:28] INFO     Created new transport with session ID:                  ]8;id=14503361;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=14503362;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py#255\255]8;;\
                             47738976609249b08bae03455ae02e89                                                      

INFO:     127.0.0.1:39434 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=14503367;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503368;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 47738976609249b08bae03455ae02e89           ]8;id=14503373;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=14503374;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=14503379;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=14503380;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

2 + 2 equals **4**.

[04/25/26 15:22:30] INFO     Terminating session: 47738976609249b08bae03455ae02e89           ]8;id=14503427;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=14503428;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http.py#785\785]8;;\

INFO:     127.0.0.1:39488 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=14503433;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503434;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     GET stream disconnected, reconnecting in 1000ms...              ]8;id=14503440;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=14503441;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py#298\298]8;;\

### Direct Tool Invocation

While tools are typically invoked by the agent based on user requests, you can also call MCP tools directly. This can be useful for workflow scenarios where you orchestrate multiple tools together.

In [18]:
query = {"x": 10, "y": 20}

with streamable_http_mcp_client:
    # direct tool invocation
    result = streamable_http_mcp_client.call_tool_sync(
        tool_use_id="tool-123", name="calculator", arguments=query
    )

    # Process the result
    print(f"Calculation result: {result['content'][0]['text']}")

[04/25/26 15:24:10] INFO     Created new transport with session ID:                  ]8;id=14503446;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=14503447;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py#255\255]8;;\
                             21e3fc39880d440bae4099c83434df8d                                                      

INFO:     127.0.0.1:52536 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=14503452;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503453;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 21e3fc39880d440bae4099c83434df8d           ]8;id=14503458;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=14503459;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=14503464;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=14503465;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=14503500;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503501;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

Calculation result: 30


                    INFO     Terminating session: 21e3fc39880d440bae4099c83434df8d           ]8;id=14503506;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=14503507;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http.py#785\785]8;;\

INFO:     127.0.0.1:52582 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=14503512;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503513;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

You can optionally also provide `read_timeout_seconds` while calling an MCP server tool to avoid it running for too long

In [19]:
with streamable_http_mcp_client:
    try:
        result = streamable_http_mcp_client.call_tool_sync(
            tool_use_id="tool-123",
            name="long_running_tool",
            arguments={"name": "Amazon"},
            read_timeout_seconds=timedelta(seconds=30),
        )

        if result["status"] == "error":
            print(f"Tool execution failed: {result['content'][0]['text']}")
        else:
            print(f"Tool execution succeeded: {result['content'][0]['text']}")
    except Exception as e:
        print(f"Tool call timed out or failed: {str(e)}")

[04/25/26 15:24:26] INFO     Created new transport with session ID:                  ]8;id=14503518;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=14503519;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http_manager.py#255\255]8;;\
                             45e6b663be6c4f6999f322ee55c79287                                                      

INFO:     127.0.0.1:56298 - "POST /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=14503524;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503525;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Received session ID: 45e6b663be6c4f6999f322ee55c79287           ]8;id=14503530;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=14503531;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py#181\181]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=14503536;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=14503537;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/client/streamable_http.py#193\193]8;;\

...

                    INFO     HTTP Request: POST http://localhost:8000/mcp "HTTP/1.1 200 OK"         ]8;id=14503572;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503573;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

Tool execution succeeded: Hello Amazon


                    INFO     Terminating session: 45e6b663be6c4f6999f322ee55c79287           ]8;id=14503578;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=14503579;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/mcp/server/streamable_http.py#785\785]8;;\

INFO:     127.0.0.1:58866 - "DELETE /mcp HTTP/1.1" 200 OK


                    INFO     HTTP Request: DELETE http://localhost:8000/mcp "HTTP/1.1 200 OK"       ]8;id=14503584;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14503585;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/httpx/_client.py#1740\1740]8;;\

### Interacting with multiple MCP servers

With Strands Agents you can also interact with multiple MCP servers using the same agent and configure tools setups such as the max number of tools that can be used in parallel (`max_parallel_tools`). Let's create a new agent to showcase this configuration:

<div style="text-align:center">
    <img src="images/architecture_3.png" width="85%" />
</div>

In this agent, we will again use the AWS Documentation MCP server and we will also use the [AWS CDK MCP Server](https://awslabs.github.io/mcp/servers/cdk-mcp-server/) which helps with AWS Cloud Development Kit (CDK) best practices, infrastructure as code patterns and security compliance with CDK Nag.

First let's connect to the two MCP servers using the stdio transport

In [20]:
# Connect to an MCP server using stdio transport
aws_docs_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

# Connect to an MCP server using stdio transport
cdk_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(command="uvx", args=["awslabs.cdk-mcp-server@latest"])
    )
)

#### Create Agent with MCP servers

Next we will create the agent with the tools from both MCP servers

In [23]:
# Create an agent with MCP tools
with aws_docs_mcp_client, cdk_mcp_client:
    # Get the tools from the MCP server
    tools = aws_docs_mcp_client.list_tools_sync() + cdk_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="amazon.nova-pro-v1:0",
        tools=tools)

    response = agent(
        "What is Amazon Bedrock pricing model. Be concise. Also what are the best practices related to CDK?"
    )

[04/25/26 15:28:59] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=14503602;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=14503603;file:///mnt/strands_agents_samples/.venv/lib/python3.10/site-packages/botocore/credentials.py#1392\1392]8;;\

<thinking> To address the user's request, I need to perform two separate searches: one for the Amazon Bedrock pricing model and another for best practices related to AWS CDK. I will use the `search_documentation` tool to find relevant documentation for both topics. </thinking>


Tool #1: search_documentation

Tool #2: search_documentation
Here's the information you requested:

**Amazon Bedrock Pricing Model:**
The Amazon Bedrock pricing model is based on the number of Compute Units (CMUs) used. The cost is calculated using CMUs, ModelCopy CloudWatch metrics, billing rates, and five-minute inference windows. For detailed cost calculation, refer to the [official documentation](https://docs.aws.amazon.com/bedrock/latest/userguide/import-model-calculate-cost.html).

**Best Practices for AWS CDK:**
The best practices for AWS CDK include securing CDK deployments using IAM roles, guardrails, permissions boundaries, Service Control Policies (SCPs), and following the least privilege grant metho

### Congratulations!

In this notebook you learned how to connect with MCP servers using Strands Agent and two MCP transport protocols: stdio and Streamable HTTP. You also learned how to connect multiple MCP servers to the same agent. Next, let's see how to use different models with your agent